In [4]:
import random
import numpy as np
import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

# 1. Generate 100 uniquely named nodes
total_nodes = 100
node_names = [f"Node_{i+1:03d}" for i in range(total_nodes)]

# Assign each node to a random metadata category/sector for filtering
sectors = ["Sector A", "Sector B", "Sector C", "Sector D"]
random.seed(42)
node_sectors = {node: random.choice(sectors) for node in node_names}

# 2. Create the graph and add nodes
G = nx.Graph()
G.add_nodes_from(node_names)

# 3. Create random varied connections
for i in range(1, total_nodes):
    target = random.choice(node_names[:i])
    G.add_edge(node_names[i], target)

all_nodes = list(G.nodes())
for _ in range(120): 
    source = random.choice(all_nodes)
    target = random.choice(all_nodes)
    if source != target:
        G.add_edge(source, target)

# 4. Calculate 3D layout positions
pos = nx.spring_layout(G, dim=3, k=0.18, seed=42)

# 5. Extract Node Coordinates
node_x, node_y, node_z, actual_names = [], [], [], []
for node in G.nodes():
    x, y, z = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_z.append(z)
    actual_names.append(node)

# 6. Extract Edge Coordinates
edge_x, edge_y, edge_z = [], [], []
for edge in G.edges():
    x0, y0, z0 = pos[edge[0]]
    x1, y1, z1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])

# 7. FIXED FOR JAGGED LINES: Using smooth transparent alpha lines to stop pixelation
edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    # Slightly reducing width and adding a softer alpha channel handles antialiasing much better in WebGL
    line=dict(color='rgba(0, 255, 204, 0.4)', width=1.5), 
    hoverinfo='none'
)

# Base default node visual state
node_trace = go.Scatter3d(
    x=node_x, y=node_y, z=node_z,
    mode='markers+text',
    text=actual_names,
    hoverinfo='text',
    textposition="top center",
    marker=dict(
        showscale=False,
        color='#00e1ff',
        size=6,
        line=dict(color='#0b0b0b', width=1.5)
    ),
    textfont=dict(color='rgba(255, 255, 255, 0.85)', size=10)
)

# 8. Define Animated States (Smooth camera orbit rotation)
frames = []
angles = np.linspace(0, 2 * np.pi, 40)
for idx, angle in enumerate(angles):
    frames.append(
        go.Frame(
            layout=dict(
                scene=dict(
                    camera=dict(
                        eye=dict(x=1.4 * np.cos(angle), y=1.4 * np.sin(angle), z=1.0)
                    )
                )
            ),
            name=f"frame_{idx}"
        )
    )

# 9. Dynamic Filter Logic for Dropdown Menu
default_colors = ['#00e1ff'] * total_nodes
default_sizes = [6] * total_nodes

menu_buttons = [
    dict(
        label="All Sectors",
        method="restyle",
        args=[{"marker.color": [default_colors], "marker.size": [default_sizes]}],
    )
]

for target_sector in sectors:
    filter_colors = []
    filter_sizes = []
    for node in actual_names:
        if node_sectors[node] == target_sector:
            filter_colors.append('#ff6a00') 
            filter_sizes.append(9)          
        else:
            filter_colors.append('rgba(80, 80, 80, 0.3)') 
            filter_sizes.append(4)          
            
    menu_buttons.append(
        dict(
            label=f"Filter: {target_sector}",
            method="restyle",
            args=[{"marker.color": [filter_colors], "marker.size": [filter_sizes]}],
        )
    )

# 10. Configure Layout with Native High-DPI and WebGL fixes
clean_axis = dict(visible=False)

fig = go.Figure(data=[edge_trace, node_trace], frames=frames)
fig.update_layout(
    title=dict(
        text="High-Quality 3D Network Engine (Anti-Aliased)",
        font=dict(color='#ffffff', size=18),
        x=0.05, y=0.95
    ),
    showlegend=False,
    margin=dict(l=0, r=0, b=0, t=0), 
    autosize=True,
    paper_bgcolor='#0b0b0b',  
    plot_bgcolor='#0b0b0b',   
    scene=dict(
        xaxis=clean_axis, yaxis=clean_axis, zaxis=clean_axis,
        bgcolor='#0b0b0b',
        camera=dict(
            eye=dict(x=1.4, y=1.4, z=1.0),
            projection=dict(type='perspective')
        )
    ),
    updatemenus=[
        dict(
            type="buttons",
            buttons=[dict(
                label="▶ Spin Animation",
                method="animate",
                args=[None, dict(
                    frame=dict(duration=50, redraw=False),
                    fromcurrent=True,
                    transition=dict(duration=40, easing="quadratic-in-out")
                )]
            )],
            direction="left",
            pad={"r": 10, "t": 85},
            showactive=False,
            x=0.05, xanchor="left",
            y=0.9, yanchor="top",
            font=dict(color="#ffffff", size=12),
            bgcolor="rgba(0, 255, 204, 0.2)",
            bordercolor="#0025ff"
        ),
        dict(
            type="dropdown",
            buttons=menu_buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.95, xanchor="right",
            y=0.95, yanchor="top",
            font=dict(color="#ffffff", size=12),
            bgcolor="#1f1f1f",
            bordercolor="#ff6a00"
        )
    ]
)

# 11. Run Render Configurations
pio.renderers.default = 'browser'

# Critical config tweak: Force the browser to render at higher pixel density
config = {
    'scrollZoom': True,           
    'displayModeBar': True,       
    'responsive': True,
    'plotGlPixelRatio': 2  # Forces Plotly to scale WebGL to 2x resolution (fixes pixelated edges)
}

fig.show(config=config)
